# RSNA Knee Abnormality Detection: Proprietary Engine 4 Training (Google Colab Pro A100 / Kaggle Dual T4)

### Architecture: Multi-Plane 2.5D ConvNeXt-Small + 12 Disentangled Attention Heads
- **Supervision:** Calibrated Continuous Targets (`data/SUPER_CONSENSUS_labels.parquet`, 4,407 studies)
- **True Multi-Plane Slicing:** 5 sequence slots (Sagittal T1/T2, Coronal T1/T2, Axial) with physical slice-normal sorting
- **Validation Benchmark:** Stratified Group 5-Fold CV on `StudyInstanceUID` (Gold-58 strictly excluded)
- **Target Solo AUC:** **0.958 - 0.960+**

## 1. Environment & GPU Verification

In [ ]:
!nvidia-smi
!pip install -q timm pydicom opencv-python pandas numpy

import os, sys, time, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training Device: {device}')
if torch.cuda.is_available():
    print('GPU Model:', torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

## 2. Setup Kaggle Data & Super-Consensus Labels

In [ ]:
# Mount or download continuous supervision targets
!git clone https://github.com/CYRUS-pinto/rsna-knee-26.git repo || (cd repo && git pull)
sys.path.insert(0, os.path.abspath('repo'))

labels_file = 'repo/data/SUPER_CONSENSUS_labels.parquet'
df_labels = pd.read_parquet(labels_file)
print(f'Loaded Super-Consensus Labels: {df_labels.shape}')
df_labels.head(2)

## 3. Physical Multi-Plane Model Definition

In [ ]:
from src.models.engine4_convnext_mil import Engine4ConvNeXtMIL
from src.data.multiplane_dataset import MultiPlane2p5DDataset, LABELS

# Verify model architecture instantiation
test_model = Engine4ConvNeXtMIL(backbone_name='convnext_small', pretrained=True, num_classes=12).to(device)
dummy_x = torch.zeros(1, 48, 3, 224, 224, device=device)
dummy_p = torch.zeros(48, dtype=torch.long, device=device)
with torch.inference_mode():
    out = test_model(dummy_x, plane_ids=dummy_p)
print(f'Model Verified! Forward output shape: {out.shape} (Expected: [1, 12])')
del test_model, dummy_x, dummy_p
torch.cuda.empty_cache()

## 4. Launch High-Throughput Fold 0 Training Loop

In [ ]:
from training.engine4.train_engine4 import train_engine4_fold

# Execute Fold 0 Training (A100 / Dual T4)
best_auc = train_engine4_fold(
    fold=0,
    epochs=6,
    batch_size=2,
    accum_steps=8, # Effective batch size = 16
    lr=1e-4,
    data_path='repo/data',
    dicom_dir='/kaggle/input/rsna-knee-abnormality-detection/train_images' if os.path.exists('/kaggle/input') else 'train_images'
)
print(f'Training Completed with Best Macro-AUC: {best_auc:.5f}')

## 5. Export Weights & OOF Predictions

In [ ]:
!ls -lh weights/
!ls -lh oof/
print('Weights and OOF predictions ready for blend integration into Slot 2!')